In [8]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated

In [9]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [10]:
def draft_email(state: State) -> dict:
    """Draft an email — this runs automatically."""
    return {"messages": ["DRAFT: Dear Boss, I'd like to request a raise based on my performance..."]}

def send_email(state: State) -> dict:
    """Send the email — we want human approval first!"""
    return {"messages": [" Email sent successfully!"]}

In [11]:
builder = StateGraph(State)
builder.add_node("draft", draft_email)
builder.add_node("send", send_email)
builder.add_edge(START, "draft")
builder.add_edge("draft", "send")
builder.add_edge("send", END)

In [12]:
memory = InMemorySaver()

In [13]:
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["send"]           
)

In [14]:
config = {"configurable": {"thread_id": "email_1"}}


result = graph.invoke(
    {"messages": [HumanMessage(content="Write an email asking for a raise")]},
    config=config
)

print(" Draft:", result["messages"][-1].content)
print("\nDo you approve sending this? (y/n)")
approval = input("> ")

if approval.lower() == "y":
    # Resume — pass None to continue from where we left off
    result = graph.invoke(None, config=config)
    print(result["messages"][-1].content)   
else:
    print(" Email cancelled by human.")

 Draft: DRAFT: Dear Boss, I'd like to request a raise based on my performance...

Do you approve sending this? (y/n)
 Email cancelled by human.
